# Practical work on SAR statistics

### Emanuele DALSASSO, Florence TUPIN and Christophe KERVAZO

Images of the practical work can be found on:
https://perso.telecom-paristech.fr/tupin/TPSAR/DATA/images/

Some useful functions are available in the file *mvalab.py*.

The practical work will be organised as follows:
* A. Single look data distribution
* B. Speckle simulation and multiplicative noise model
* Image despeckling: simple averaging and Lee filter
* Nonlocal denoising with NL-means

### First steps
The first three cells will download the *mvalab.py* stript with some useful functions to read the data, import the libraries needed and implement a fuction to visualize SAR images in a more pleasant way, as well as creating synthetic speckle noise.
  

In [ ]:
!wget https://partage.imt.fr/index.php/s/BqjLCyif5LMxZm7/download/mvalab.py
!wget https://partage.imt.fr/index.php/s/TG6yrKyHfKDmM8e/download/lely.npy
!wget https://partage.imt.fr/index.php/s/QfAm3sTXNrnBX8t/download/pepper.png

In [ ]:
%matplotlib inline
import scipy
from scipy import signal
import scipy.signal
import scipy as spy
import scipy.fftpack
from scipy import ndimage
from scipy import special
from scipy import ndimage
import numpy as np
import math
import matplotlib.pyplot as plt
import mvalab as mvalab
from urllib.request import urlopen
import cmath
from bokeh.plotting import figure, output_file, show
from bokeh.plotting import show as showbokeh
from bokeh.io import output_notebook

output_notebook()
from bokeh.colors import Color as bcolor

# from bokeh.colors.rgb import RGB
from bokeh.colors import RGB
from mvalab import visusar

plt.rcParams["figure.figsize"] = [8, 8]
plt.rcParams["figure.max_open_warning"] = 30

In [ ]:
def visusar_colab(im, size_fig=500, title="", threshold=None):  # special colab
    liste_couleurs_grises = []
    for k in range(256):
        liste_couleurs_grises.append(RGB(k, k, k))

    """
    by default: compute mean and std on the image at hand
    if 0: display full dynamic
    else: use the custom value that is provided
    """
    if threshold == None:
        threshold = np.mean(im) + 3 * np.std(im)
    elif threshold == 0:
        threshold = np.max(im)
    else:
        threshold = threshold

    def normalise_image_pour_bokeh(X, threshold):
        imt = np.copy(X.copy())
        imt = imt - np.min(imt)
        imt /= threshold
        imt[imt < 0] = 0
        imt[imt > 1] = 1

        imt *= 255
        imt = imt.astype(np.uint8)
        return imt

    img = normalise_image_pour_bokeh(
        np.flipud(im), threshold
    )  # np.flipud(np.fliplr(im)))
    title = title + " : mean = " + str(np.mean(im)) + " std = " + str(np.std(im))
    p = figure(
        tooltips=[("x", "$x{int}"), ("y", "$y{int}"), ("value", "@image")],
        title=title,
        y_range=[im.shape[0], 0],
        x_range=[0, im.shape[1]],
        width=size_fig,
        height=int(size_fig * im.shape[0] / im.shape[1]),
        match_aspect=True,
    )

    # must give a vector of images
    p.image(
        image=[img],
        palette=liste_couleurs_grises,
        x=0,
        y=im.shape[0],
        dw=im.shape[1],
        dh=im.shape[0],
        dilate=False,
    )
    showbokeh(p)


def injectspeckle_amplitude(img, L):
    rows = img.shape[0]
    columns = img.shape[1]
    s = np.zeros((rows, columns))
    for k in range(0, L):
        gamma = (
            np.abs(np.random.randn(rows, columns) + np.random.randn(rows, columns) * 1j)
            ** 2
            / 2
        )
        s = s + gamma
    s_amplitude = np.sqrt(s / L)
    ima_speckle_amplitude = np.multiply(img, s_amplitude)
    return ima_speckle_amplitude


def read_png(filename):
    ima = plt.imread(filename)
    ima = np.array(ima)
    ima = np.sqrt(ima[:, :, 0] ** 2 + ima[:, :, 1] ** 2 + ima[:, :, 2] ** 2)
    return ima

## A. Single look data distributions
In this part, we will use a Single Look Complex (SLC) image and analyze the Probability Density Function (PDF) on a manually selected homogeneous area to verify Goodman's hypothesis.

The image has been acquired by the Sentinel-1 sensor over the Lelystad zone in the Netherlands (flat area with fields crops, water and urban areas).
Vizualize the amplitude image and interpret it. You may want to have a look to an [optical image](https://goo.gl/maps/JJcYcRjMKj1p6uqW8) of the area

***N.B.***: An amplitude image is given by the modulus of the electro-magnetic field and intensity is the square of the amplitude (proportional to the signal power).  

In [ ]:
pageweb = "https://perso.telecom-paristech.fr/tupin/TPSAR/pilelely/"
image = "Lely.CXF"
im_slc_senti_lely_liste = mvalab.imz2mat(pageweb + image)
im_slc_senti_lely = im_slc_senti_lely_liste[0]
ncol = im_slc_senti_lely_liste[1]
nlig = im_slc_senti_lely_liste[2]

# plot the amplitude image

visusar(im_slc_senti_lely)  # - insert code here to display the image

# because the image is big you can display a crop of 1024 x 1024 pixels

visusar(np.abs(im_slc_senti_lely)[:1024, :1024])

### A.1.: Data distributions for an homogeneous area
- Select a **physically homogeneous** area and compute the distribution of the intensity, amplitude, phase, real part and imaginary part. Some useful functions are:
  - `np.angle`
  - `np.real`
  - `np.imag`

- Compute the histograms for these variables and draw on the histogram the theoretical curve by computing the mean value $\mu$ and the standard deviation $\sigma$ with the function `scipy.stats.[distribution].fit` by replacing `[distribution]` with the [expected pdf](https://docs.scipy.org/doc/scipy/reference/stats.html)

- Compute the coefficient of variation $\gamma=\frac{\sigma}{\mu}$ in amplitude and in intensity.

***N.B.***: The Goodman model is valid only on homogeneous area. That is why it is important to select pixels sharing the same distribution (with the same underlying reflectivity).


In [ ]:
# Select a crop of at least 200 by 200 pixels on a homogeneous area
crop_slc = im_slc_senti_lely[850:1000, 0:200]  # complete
print(crop_slc.shape)
visusar(np.abs(crop_slc))

# Compute amplitude, intensity, phase, real and imaginary part
amp_senti_lely = np.abs(crop_slc)  # complete
int_senti_lely = np.abs(crop_slc) ** 2  # complete
ph_senti_lely = np.angle(crop_slc)  # complete
real_senti_lely = np.real(crop_slc)  # complete
imag_senti_lely = np.imag(crop_slc)  # complete

In [ ]:
# Plot the histograms and verify they match the theoretical distribution
plt.rcParams["figure.figsize"] = [8, 8]

plt.figure()
_, bins, _ = plt.hist(
    amp_senti_lely.ravel(), bins="auto", density=True, range=[0.0, 100]
)
mu, sigma = scipy.stats.rayleigh.fit(amp_senti_lely, method="MM")  # Complete
best_fit_line = scipy.stats.rayleigh.pdf(bins, mu, sigma)  # Complete
plt.plot(bins, best_fit_line)
plt.title("histogram of amplitude")
plt.show()
print("mu", mu, "sigma", sigma)

plt.figure()
_, bins, _ = plt.hist(
    int_senti_lely.ravel(), bins="auto", density=True, range=[0.0, 5000]
)
mu, sigma = scipy.stats.expon.fit(int_senti_lely)  # Complete
best_fit_line = scipy.stats.expon.pdf(bins, mu, sigma)  # Complete
plt.plot(bins, best_fit_line)
plt.title("histogram of intensity")
plt.show()

plt.figure()
plt.hist(
    ph_senti_lely.ravel(), bins="auto", density=True, range=[-np.pi, np.pi]
)  # Which distribution?
plt.title("histogram of phase")
plt.show()

plt.figure()
_, bins, _ = plt.hist(
    real_senti_lely.ravel(), bins="auto", density=True, range=[-100, 100]
)
mu, sigma = scipy.stats.norm.fit(real_senti_lely)  # Complete
best_fit_line = scipy.stats.norm.pdf(bins, mu, sigma)  # Complete
plt.plot(bins, best_fit_line)
plt.title("histogram of real part")
plt.show()

plt.figure()
_, bins, _ = plt.hist(
    imag_senti_lely.ravel(), bins="auto", density=True, range=[-100, 100]
)
mu, sigma = scipy.stats.norm.fit(imag_senti_lely)  # Complete
best_fit_line = scipy.stats.norm.pdf(bins, mu, sigma)  # Complete
plt.plot(bins, best_fit_line)
plt.title("histogram of imaginary part")
plt.show()

In [ ]:
# Compute the coefficient of variation on the homogeneous crop using intensity data
m_I = np.mean(amp_senti_lely)  # complete
sigma_I = np.std(amp_senti_lely)  # complete
coeff_var_I = m_I / sigma_I  # complete
print("Coefficient of variation in intensity: " + str(coeff_var_I))

# Compute the coefficient of variation on the homogeneous crop using amplitude data
m_A = np.mean(int_senti_lely)  # complete
sigma_A = np.std(int_senti_lely)  # complete
coeff_var_A = m_I / sigma_I  # complete
print("Coefficient of variation in amplitude: " + str(coeff_var_A))

### Question A.1.
What are the distributions followed by real part, imaginary part, phase, intensity and amplitude on an homogeneous area ?
What are the values for the coefficient of variation for amplitude and intensity data? Are they in accordance with the theoretical values ?

### Answer A.1.
...
__answer__:
The following statistics of the signals follows:
- a rayleigh distribution for the amplitude
- a exponential law for the the Intensity
- a Uniform law for the phase
- a Normal law for the Real and Imaginary part

Regarding the coefficient of variation for amplitude and intensity data, they are in accordance with the theoretical values, as the bins closely align with the fitted distribution curve.

### A.2. Local analysis of the coefficient of variation

The coefficient of variation $\gamma=\frac{\sigma}{\mu}$ (standard deviation normalized by the mean) is an indication of the local homogeneity of the scene.
It can be computed locally around each pixel using a moving window.

Using 2D convolution (`signal.convolve2d(..,..,mode='same')`) to speed up the processing, compute the image of the coefficient of variation by taking the intensity of `im_slc_senti_lely`.

In [ ]:
X = im_slc_senti_lely
mean = signal.convolve2d(X, np.ones((7, 7)) / (7 * 7), mode="same")
variance = signal.convolve2d(np.multiply(X, X), np.ones((7, 7)) / (7 * 7), mode="same")
std = np.sqrt(variance - np.multiply(mean, mean))

Var = np.divide(std, mean + 1e-6)
print(Var)
# show the image

### Question A.2.
Comment the results of the image of local coefficient of variation and local standard deviation.
- Which structures of the image are highlighted with the coefficient of variation ?
- What is the influence of the window size ?
- Why is the local standard deviation not adapted to measure the local homogeneity of the scene ?


### Answer A.2.
...
__answer__:


-> The coefficient of variation highlights edges and heterogeneous areas of the image, as it normalizes the standard deviation by the local mean, making it more sensitive to variations in contrast.
->
The window size influences the results by smoothing or enhancing local variations.  Edges are more visible with higher windows size.
-> The local standard deviations increase with the mean of the signal, that's why we have to substract to have a more robust model.

In [ ]:
# function to compute the coefficient of variation
def compute_coeff_var(ima_int, size_window):
    # create the average window
    mask = np.ones((size_window, size_window)) / (size_window * size_window)

    # compute the mean image E{I}
    ima_int_mean = signal.convolve2d(ima_int, mask, mode="same")  # Complete, E{I}

    # compute the variance image (var{I} = E{I^2} - E{I}^2)
    ima_int_square = np.multiply(ima_int, ima_int)  # I^2
    ima_int_mean_square = signal.convolve2d(
        ima_int_square, mask, mode="same"
    )  # complete # E{I^2}
    ima_variance = ima_int_mean_square - np.multiply(
        ima_int_mean, ima_int_mean
    )  # var{I}

    # compute coefficient of variation
    ima_coeff_var = np.divide(
        np.sqrt(ima_variance), ima_int_mean + 1e-6
    )  # Complete, avoid division by 0
    return ima_variance, ima_int_mean, ima_coeff_var

In [ ]:
# take the intensity
ima_int = np.abs(np.multiply(im_slc_senti_lely, np.conjugate(im_slc_senti_lely)))
visusar_colab(np.sqrt(ima_int))

ima_variance, _, ima_coeff_var = compute_coeff_var(ima_int, 7)

# plot the coefficient of variation
visusar_colab(ima_coeff_var)
# plt.title('Coefficient of variation')

# plot the standard deviation
visusar_colab(np.sqrt(ima_variance), threshold=20000)
# plt.title('Standard deviation')

In [ ]:
for window in [3, 7, 10, 20]:
    print("plot for window ", window)
    ima_variance, _, ima_coeff_var = compute_coeff_var(ima_int, 7)

    # plot the coefficient of variation
    visusar_colab(ima_coeff_var)
    # plt.title('Coefficient of variation')

    # plot the standard deviation
    visusar_colab(np.sqrt(ima_variance), threshold=20000)

## B. Speckle simulation and multiplicative noise model
Speckle phenomenon is a determined by the physics of the ground and is a deterministic process. It is however common to model is as a multiplicative noise. You can use the function `injectspeckle_amplitude(img,L)` to simulate speckle noise with a given number of looks L on an amplitude image.

When processing noisy SAR images, it is often beneficial to move into the log domain. This way, the variance is stabilized (i.e. it becomes homogeneous on the whole image).

Moreover, the distribution approaches a Gaussian distribution as L increases.

### B.1. Edge detection
Create a simulated image by concatenating a dark rectangle with a bright one and simulate speckle noise on it (you can use values of 100 and 150 for the two parts). Use the canny edge detector and the sobel edge detector to process the amplitude image and the log of the amplitude.

In [ ]:
bright_rectangle = 150
dark_rectangle = 100
simu_image = np.concatenate(
    (dark_rectangle * np.ones((512, 256)), bright_rectangle * np.ones((512, 256))),
    axis=1,
)  # complete

# use the injectspeckle_amplitude function to create speckle with L=1
simu_speckle = injectspeckle_amplitude(simu_image, 1)  # complete
plt.figure()
plt.imshow(simu_speckle, cmap="gray")
plt.show()

# use pepper.png image to create a spckled image with L=1
import imageio

pepper = imageio.imread("pepper.png")
pepper = pepper[:, :, 1]
simu_speckle2 = injectspeckle_amplitude(pepper, 1)
visusar(simu_speckle2)
plt.title("simu pepper")

In [ ]:
def sobel_filters(img):
    Kx = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32)
    Ky = np.array([[1, 2, 1], [0, 0, 0], [-1, -2, -1]], np.float32)

    Ix = ndimage.filters.convolve(img, Kx)
    Iy = ndimage.filters.convolve(img, Ky)

    G = np.hypot(Ix, Iy)
    G = G / G.max() * 255
    theta = np.arctan2(Iy, Ix)

    return (G, theta)


def canny_filters(img):
    Kx = np.array([[-1 / 2, 0, 1 / 2]], np.float32)
    Ky = np.transpose(Kx)

    Ix = ndimage.filters.convolve(img, Kx)
    Iy = ndimage.filters.convolve(img, Ky)

    G = np.hypot(Ix, Iy)
    G = G / G.max() * 255
    theta = np.arctan2(Iy, Ix)

    return (G, theta)


def threshold(img, lowThresholdRatio=0.05, highThresholdRatio=0.08):
    highThreshold = img.max() * highThresholdRatio
    lowThreshold = highThreshold * lowThresholdRatio
    M, N = img.shape
    res = np.zeros((M, N), dtype=np.int32)

    weak = np.int32(25)
    strong = np.int32(255)

    strong_i, strong_j = np.where(img >= highThreshold)
    zeros_i, zeros_j = np.where(img < lowThreshold)

    weak_i, weak_j = np.where((img <= highThreshold) & (img >= lowThreshold))

    res[strong_i, strong_j] = strong
    # res[weak_i, weak_j] = weak

    return res

In [ ]:
# apply filters in intensity
edge_n, theta = sobel_filters(simu_speckle2)
res = threshold(edge_n)
plt.figure()
plt.title("sobel filter")
plt.imshow(res, cmap="gray")

edge_n, theta = canny_filters(simu_speckle2)
res = threshold(edge_n)
plt.figure()
plt.title("canny filter")
plt.imshow(res, cmap="gray")
plt.show()

In [ ]:
# apply filters to log-transformed data
edge_n, theta = sobel_filters(np.log(simu_speckle2 + 1e-6))  # complete
res = threshold(edge_n)
plt.figure()
plt.imshow(res, cmap="gray")

edge_n, theta = canny_filters(np.log(simu_speckle2 + 1e-6))  # complete
res = threshold(edge_n)
plt.figure()
plt.imshow(res, cmap="gray")
plt.show()

### Question B.1.
Compare the two results. What can you observe? Why do you have more false positives on the amplitude image?

### Answer B.1.
The Sobel filter provides sharper edges and better noise suppression, whereas the Canny filter retains more noise.

In the amplitude image (without the log transformation), both filters struggle to accurately detect the edges due to the presence of speckle noise. Applying the log transformation enhances contrast in low-intensity areas, making the edges more distinguishable.

This effect can be explained by the nature of speckle noise, which is multiplicative. The log transformation converts it into an additive noise, reducing its relative impact and improving edge detection.


## C. Image despeckling: simple averaging and Lee filter

The local coefficient of variation is also used in a very famous filter for SAR images: the Lee filter.
The principle of the filter is to combine the pixel value $I_s$ (intensity value of pixel $s$) and the local mean $\hat{\mu}_{s}$ depending on the local coefficient of variation $\hat{\gamma}_s$ with the following formula :
$
  \hat{I}_s= \hat{\mu}_{s}+k_s (I_s-\hat{\mu}_{s})
$

and
$
  k_s=1- \frac{\gamma_{Sp}^2}{\hat{\gamma}_s^2}
$

$\gamma_{Sp}$ is the theoretical value of the coefficient of variation for a pure speckle ($\gamma_{Sp}=\frac{1}{\sqrt{L}}$ for a L-look intensity image).

In this part, we will work both on a noisless image with simulated speckle and on a real image.

### Question C.1.
Using the previously implemented `compute_coeff_var` function, implement the Lee filter and visualize the denoised image. Comment the result and compare it with a local mean.

Warning : $k$ should be in $[0,1]$.

__answer__:
In the original image, speckle noise is prominent, making it difficult to distinguish structures.

-> The Lee filter effectively reduces noise while preserving bright areas, which often correspond to buildings or other strong reflectors. However, in some cases, it can introduce artifacts, creating misleading structures in the image.
-> The local mean filter, in contrast, over-smooths the image, leading to a blurry appearance and loss of detail due to excessive averaging.

When applied to the second image, the Lee filter introduces artifacts, such as the appearance of non-existent structures, making interpretation unreliable. In this case, the local mean filter provides a more robust but with less performance-wise result.

In [ ]:
noiseless = np.load("./lely.npy")
visusar(noiseless)

noisy = injectspeckle_amplitude(noiseless, 1)
visusar(noisy)

In [ ]:
ima_int = np.square(noisy)  # take the intensity
_, ima_int_mean, ima_coeff_var = compute_coeff_var(ima_int, 7)

# compute ks
gamma_s = ima_coeff_var
gamma_SP = 1 / np.sqrt(7)
ks = 1 - np.divide(gamma_SP, gamma_s)  # complete

# force k to have values comprised in the range [0,1]
ks[ks < 0] = 0
visusar(ks)

# filter the image
image_lee_filtered = ima_int_mean + np.multiply(ks, ima_int - ima_int_mean)  # complete
visusar(np.sqrt(image_lee_filtered))
plt.title("Image denoised using Lee filter")
visusar(np.sqrt(ima_int_mean))
plt.title("Image denoised with local averaging")

In [ ]:
# Real SLC SAR image
real_noisy_slc = mvalab.imz2mat(
    "https://perso.telecom-paristech.fr/tupin/TPSAR/pilelely/multitemp/lely_tuple_multitemp.IMA"
)
real_noisy_int = np.square(np.abs(real_noisy_slc[0][:, :, 0]))
visusar(np.sqrt(real_noisy_int))

In [ ]:
_, ima_int_mean, ima_coeff_var = compute_coeff_var(real_noisy_int, 7)

# compute ks
gamma_s = ima_coeff_var
gamma_SP = 1 / np.sqrt(7)
ks = 1 - np.divide(gamma_SP, gamma_s)  # complete

# force k to have values comprised in the range [0,1]
ks[ks < 0] = 0
visusar(
    ks,
)

# filter the image
image_lee_filtered = ima_int_mean + np.multiply(
    ks, real_noisy_int - ima_int_mean
)  # complete
visusar(np.sqrt(image_lee_filtered))
plt.plot("Image denoised using Lee filter")
visusar(np.sqrt(ima_int_mean))
plt.title("Image denoised with local averaging")

## D. Nonlocal denoising with NL-means
When moving into the log domain, beside variance stabilization, the noise becomes additive and roughly gaussian as L increases.

![fishertippett_gaussian](https://www.mdpi.com/remotesensing/remotesensing-12-02636/article_deploy/html/images/remotesensing-12-02636-g003-550.jpg)

Log-transformed speckle follows a so-called Fisher-Tippett distribution with:
- $\sigma^2 = \psi(1,L)$
- $\mu = \psi(L)-log(L)$

In the following, we will use the `skimage.restoration` [python library](https://scikit-image.org/docs/dev/auto_examples/filters/plot_nonlocal_means.html) to apply the [NL-means algorithm](https://ieeexplore.ieee.org/abstract/document/1467423?casa_token=eJEGHlMBXvkAAAAA:ZiJZdw2T1qgUbwol3mXI9d17foUUfO6_jGqqkMOkKWUoefusUZo328grE1LCm94oFQZtBWs) to suppress speckle from SAR images. Namely, we will simulate speckle on a noiseless image and we will:
- see the advantage of the log-transform
- analyse the bias when averaging log-transformed samples and correct it
- compare the results of NL-means when adding Gaussian noise with the same standard deviation
- compare the results with a real SAR image with spatially correlated speckle noise

In [ ]:
from skimage.restoration import denoise_nl_means

In [ ]:
def stats_log(L):
    sigma_speckle = np.sqrt(special.polygamma(1, L))  # complete
    mean_speckle = special.psi(L) - np.log(L)  # complete
    return mean_speckle, sigma_speckle


patch_kw = dict(
    patch_size=5,  # 5x5 patches
    patch_distance=6,  # 13x13 search area
)

In [ ]:
noisy = injectspeckle_amplitude(noiseless, 1)
visusar(noisy)

# --- DENOISING IN INTENSITY ---
noisy_int = np.square(noisy)

# we use the parameter sigma_I previously estimated on a homogeneous area
denoised_int = denoise_nl_means(noisy_int, h=sigma_I, fast_mode=True, **patch_kw)
visusar(np.sqrt((denoised_int)))

# --- DENOISING IN LOG-INTENSITY ---
mean_speckle, sigma_speckle = stats_log(1)
noisy_log = np.log(noisy_int + 1e-6)  # Do a log transform of the intensity
denoised_log = denoise_nl_means(
    noisy_log, h=0.8 * sigma_speckle, fast_mode=True, **patch_kw
)
# apply bias correction
denoised_log = denoised_log - mean_speckle
visusar(np.sqrt(np.exp((denoised_log))))

In [ ]:
# Here, we add gaussian noise using the function np.random.randn(d0,d1) with the correct sigma
gaussian_noise = np.random.randn(noiseless.shape[0], noiseless.shape[1]) * sigma_speckle
noisy_gaussian = np.log(np.square(noiseless) + 1e-6) + gaussian_noise
denoised_gaussian = denoise_nl_means(
    noisy_gaussian, h=0.8 * sigma_speckle, fast_mode=True, **patch_kw
)  # complete with sigma

visusar(np.sqrt(np.exp((denoised_gaussian))))
visusar(np.sqrt(np.exp((denoised_log))))

### Question D.1.
Why do we observe such great differences when applying NL-means on an image with speckle noise and on an image corrupted with Gaussian noise?

### Answer D.1.

__answer__:  
he NL-means algorithm works by comparing patches centered on each pixel and averaging similar patches to reduce noise.

 NL-means is effective because Gaussian noise follows an additive and independent distribution. The similarity between patches is well preserved, allowing effective denoising.
 NL-means struggles because speckle noise is multiplicative, meaning its intensity depends on the underlying signal rather than being uniformly distributed. Since the noise pattern does not follow a normal distribution, patch similarity measures become less reliable, reducing the effectiveness of it
Thus, NL-means performs better on Gaussian noise but is less effective on speckle noise due to the non-additive nature of the latter.

In [ ]:
# add gaussian noise with L=20 on image 'noiseless'
L = 20
noisy_amp = injectspeckle_amplitude(noiseless, L)
noisy_int = np.square(noisy_amp)
noisy_log = np.log(noisy_int + 1e-6)

mean_speckle, sigma_speckle = stats_log(L)
gaussian_noise = np.random.randn(noiseless.shape[0], noiseless.shape[1]) * sigma_speckle
noisy_gaussian = np.log(np.square(noiseless) + 1e-6) + gaussian_noise
denoised_gaussian = denoise_nl_means(
    noisy_gaussian, h=0.8 * sigma_speckle, fast_mode=True, **patch_kw
)
denoised_speckle = denoise_nl_means(
    noisy_log, h=0.8 * sigma_speckle, fast_mode=True, **patch_kw
)

visusar(noisy_amp)
visusar(np.sqrt(np.exp(noisy_gaussian)))
visusar(np.sqrt(np.exp((denoised_gaussian))))
print("visusard with denoised speckmle")
visusar(np.sqrt(np.exp((denoised_speckle - mean_speckle))))
print("with not removing the mean")
visusar(np.sqrt(np.exp((denoised_speckle))))

### Question D.2.
Do you observe any differences between the results?
What does it happen if you do not correct the bias at the algorithm's output? You can have a look at the statistics printed when visualizing the images.

### Answer D.2.
If we do not correct the bias in the algorithm's output, there is no notable visual difference in the image. This suggests that the mean intensity is already close to zero, minimizing the perceptual impact of the bias.
...

In [ ]:
# comparison with real image
real_noisy_log = np.log(real_noisy_int + 1e-6)
mean_speckle, sigma_speckle = stats_log(1)

denoised_real = denoise_nl_means(
    real_noisy_log, h=0.8 * sigma_speckle, fast_mode=True, **patch_kw
)
print("denoised real")
visusar(np.sqrt(np.exp((denoised_real - mean_speckle))))
visusar(np.sqrt(np.exp((denoised_log))))

### Question D.3.
Comment the differences between the result on the real image and the result on the simulated one.


### Answer D.3.
...
__answer__:
The real image exhibits significantly more noise compared to the simulated image.

The simulated image has been denoised more effectively because the filter was designed for a specific noise model, which the simulated data closely follows.
In contrast, the real image does not perfectly adhere to the assumed noise distribution, leading to incomplete denoising in some regions.

## BONUS:
Most recent denoising approaches rely on deep neural networks. As you have seen, simulated speckle noise does not perfectly fit the statistics of real SAR images. In particular, the spatial correlation is not modeled by Goodman's model and this is a source of artifacts.

The **SAR2SAR** algorithm has been trained in a semi-supervised way, learning speckle statistics directly from real images. You can test it on single look images using the notebook `` SAR2SAR_Single_Look_test.ipynb`` that you can find at [this Gitlab repository](https://gitlab.telecom-paris.fr/ring/sar2sar).


### Bonus question
Comment the main differences with respect to the Lee filter, the mean filter and the NL-mean algorithm.




### Bonus answer
Main differences:

Unlike the Lee filter, mean filter, and NL-means, which rely on mathematical models for noise reduction, SAR2SAR is trained using a semi-supervised approach and learns speckle noise patterns directly from real SAR images.

-> SAR2SAR denoises SAR images without needing ground truth by learning from noisy samples.

Based on visual comparisons in the referenced article, the SAR2SAR algorithm preserves significantly more local details than traditional filters.
